# Module 06 — Exercises

These exercises rebuild the pieces of an automatic-differentiation system: finite differences,
forward mode with dual numbers, a reverse sweep by hand, topological sorting, the closures of a scalar
autograd engine, the matrix backward rules, a fused classification loss, a gradient checker, and a
planted bug to find. The mini-project trains a network with *your* engine. Only NumPy, `math` and
matplotlib are allowed — no PyTorch, no JAX. The concept guide (`README.md`) has every formula you need.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Finite differences and the best step size  `★☆☆`

Implement the two finite-difference estimators of $f'(x)$,

$$D^{+}_h f(x) = \frac{f(x+h) - f(x)}{h}, \qquad D^{c}_h f(x) = \frac{f(x+h) - f(x-h)}{2h},$$

as `forward_diff(f, x, h)` and `central_diff(f, x, h)` (both return a float), and `best_h(f, x, exact, hs, method)`,
which returns the entry of the NumPy array `hs` for which `method(f, x, h)` is closest to the known value `exact`.

Then answer (the checks hash your numbers): for $f(x) = x^3$ at $x = 2$ with $h = 0.1$, what do the two estimators return?

In [ ]:
def forward_diff(f, x, h):
    # YOUR CODE HERE
    return None

def central_diff(f, x, h):
    # YOUR CODE HERE
    return None

def best_h(f, x, exact, hs, method):
    # return the h in hs (a NumPy array) whose estimate method(f, x, h) is closest to exact
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
cube = lambda t: t ** 3
try:
    f_est = forward_diff(cube, 2.0, 0.1)
    c_est = central_diff(cube, 2.0, 0.1)
except Exception:
    f_est = c_est = None
check_hash('forward estimate of (x^3)\' at 2 with h=0.1', f_est, '8326acae5fc56719')
check_hash('central estimate of (x^3)\' at 2 with h=0.1', c_est, '58023dc1612024dd')
try:
    hs = np.logspace(-12, -1, 23)
    bh_f = best_h(np.exp, 1.0, np.e, hs, forward_diff)
    bh_c = best_h(np.exp, 1.0, np.e, hs, central_diff)
except Exception:
    bh_f = bh_c = None
check('best forward-difference h for exp at 1 lies in [1e-9, 1e-7]', bh_f is not None and 1e-9 <= bh_f <= 1e-7)
check('best central-difference h for exp at 1 lies in [1e-7, 1e-4]', bh_c is not None and 1e-7 <= bh_c <= 1e-4)
check('central beats forward at h=1e-3 for sin at 0.5',
      (abs(central_diff(np.sin, 0.5, 1e-3) - np.cos(0.5)) < abs(forward_diff(np.sin, 0.5, 1e-3) - np.cos(0.5))) if c_est is not None else False)

## Exercise 2 — Forward mode with dual numbers  `★☆☆`

Complete the `Dual` class: a value `val` carrying a tangent `dot`. Addition is given. Implement the product rule in
`__mul__`, the quotient rule in `__truediv__`, and `exp`, `sin`, `cos` via $g(a, \dot a) = (g(a), g'(a)\,\dot a)$.
Then implement `derivative(f, x)` which returns $f'(x)$ from **one** forward sweep (seed the input with tangent 1).

The hidden check asks for $\frac{d}{dx}\Big[\frac{x\,e^{x}\sin x}{x + 1}\Big]$ at $x = 2.1$.

In [ ]:
class Dual:
    def __init__(self, val, dot=0.0):
        self.val, self.dot = float(val), float(dot)
    @staticmethod
    def lift(o):
        return o if isinstance(o, Dual) else Dual(o)
    def __add__(self, o):
        o = Dual.lift(o); return Dual(self.val + o.val, self.dot + o.dot)
    def __mul__(self, o):
        o = Dual.lift(o)
        # YOUR CODE HERE (product rule)
        return Dual(self.val * o.val, 0.0)
    def __truediv__(self, o):
        o = Dual.lift(o)
        # YOUR CODE HERE (quotient rule)
        return Dual(self.val / o.val, 0.0)
    def exp(self):
        # YOUR CODE HERE
        return Dual(math.exp(self.val), 0.0)
    def sin(self):
        # YOUR CODE HERE
        return Dual(math.sin(self.val), 0.0)
    def cos(self):
        # YOUR CODE HERE
        return Dual(math.cos(self.val), 0.0)
    __radd__, __rmul__ = __add__, __mul__

def derivative(f, x):
    # f takes and returns a Dual; return f'(x) using one forward-mode sweep
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    d_prod = derivative(lambda x: x * x * x, 2.0)                      # 3x^2 = 12
    d_quot = derivative(lambda x: x.sin() / x, 2.0)                    # (x cos x - sin x)/x^2
    d_exp = derivative(lambda x: (x * 3.0).exp(), 0.5)                 # 3 e^{1.5}
    d_hidden = derivative(lambda x: x * x.exp() * x.sin() / (x + 1.0), 2.1)
except Exception:
    d_prod = d_quot = d_exp = d_hidden = None
check_close('product rule: d/dx x^3 at 2', d_prod, 12.0)
check_close('quotient rule: d/dx sin(x)/x at 2', d_quot, (2 * np.cos(2) - np.sin(2)) / 4)
check_close('chain through exp: d/dx e^(3x) at 0.5', d_exp, 3 * np.exp(1.5))
check_hash('hidden: derivative of x e^x sin x / (x+1) at 2.1', d_hidden, '41c8f0f3a88976b7')

## Exercise 3 — A reverse sweep by hand  `★☆☆`

Consider the graph with inputs $x_1 = 2$, $x_2 = 3$ and nodes

$$v_1 = x_1 x_2, \qquad v_2 = \exp(v_1 / 2), \qquad v_3 = v_1 + v_2\, x_1^2 .$$

Note that $x_1$ feeds two nodes ($v_1$ and the factor $x_1^2$) and $v_1$ feeds two nodes ($v_2$ and $v_3$ directly).
Fill in the adjoints $\bar v_3, \bar v_2, \bar v_1, \bar x_1, \bar x_2$ (as floats) by one reverse sweep,
summing contributions where a node has several children. Use the forward values already computed in the cell.

In [ ]:
# forward values (given)
v1 = 2.0 * 3.0
v2 = math.exp(v1 * 0.5)
v3 = v1 + v2 * 2.0 ** 2
# YOUR CODE HERE: fill in the adjoints (floats) in reverse topological order
bar_v3 = None
bar_v2 = None
bar_v1 = None
bar_x1 = None
bar_x2 = None
adjoints = {"v3": bar_v3, "v2": bar_v2, "v1": bar_v1, "x1": bar_x1, "x2": bar_x2}

**Check your answer:**

In [ ]:
def _get(k):
    try:
        v = adjoints[k]
        return None if v is None else float(v)
    except Exception:
        return None
check_close('adjoint of the output is 1', _get('v3'), 1.0)
check_close('adjoint of v2 is x1^2 = 4', _get('v2'), 4.0)
check_hash('hidden: adjoint of v1 (two contributions)', _get('v1'), 'a9b73619cee75fed')
check_hash('hidden: adjoint of x1 (two contributions)', _get('x1'), 'f66d0356f8092a4d')
check_hash('hidden: adjoint of x2', _get('x2'), '25da0bd92157d56a')

## Exercise 4 — Topological sort of a computational graph  `★★☆`

A graph is given as a list `nodes` and a dict `parents` mapping each node to a tuple of its parents (leaves have
no entry or an empty tuple). Implement `topo_sort(nodes, parents)` returning a list that contains every node exactly
once with each node *after* all of its parents (depth-first search with a visited set, as in the `Value.backward`
method), and `is_valid_topo(order, nodes, parents)` returning `True` iff `order` is such a list. Your function must
handle nodes that are reachable from several children (diamonds) and must not rely on the input order of `nodes`.

In [ ]:
def topo_sort(nodes, parents):
    # nodes: list of hashable node ids; parents: dict node -> tuple of parent ids (leaves may be absent)
    # return a list containing every node exactly once, each after all of its parents
    # YOUR CODE HERE
    return None

def is_valid_topo(order, nodes, parents):
    # True iff order is a permutation of nodes in which every parent precedes its child
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
chain_nodes = ["x", "a", "b", "L"]
chain_par = {"a": ("x",), "b": ("a",), "L": ("b",)}
diamond_nodes = ["L", "v3", "v2", "v1", "x1", "x2"]          # deliberately scrambled
diamond_par = {"v1": ("x1", "x2"), "v2": ("v1",), "v3": ("v1", "x1"), "L": ("v2", "v3")}
try:
    o1 = topo_sort(chain_nodes, chain_par)
    o2 = topo_sort(diamond_nodes, diamond_par)
except Exception:
    o1 = o2 = None
check('chain graph sorted', o1 == ["x", "a", "b", "L"])
def _valid(order, nodes, parents):
    if order is None or sorted(order) != sorted(nodes):
        return False
    pos = {n: i for i, n in enumerate(order)}
    return all(pos[p] < pos[n] for n in nodes for p in parents.get(n, ()))
check('diamond graph: every node once, parents first', _valid(o2, diamond_nodes, diamond_par))
try:
    ok_valid = is_valid_topo(["x1", "x2", "v1", "v2", "v3", "L"], diamond_nodes, diamond_par)
    ok_invalid = is_valid_topo(["x1", "v1", "x2", "v2", "v3", "L"], diamond_nodes, diamond_par)
    ok_dup = is_valid_topo(["x1", "x2", "v1", "v1", "v2", "L"], diamond_nodes, diamond_par)
except Exception:
    ok_valid = ok_invalid = ok_dup = None
check('is_valid_topo accepts a valid order', ok_valid is True)
check('is_valid_topo rejects a parent after its child', ok_invalid is False)
check('is_valid_topo rejects duplicates / missing nodes', ok_dup is False)
# random DAGs: generated so that parents always have smaller ids
ok_random = True
for trial in range(5):
    n = 12
    par = {i: tuple(int(p) for p in rng.choice(i, size=min(i, 2), replace=False)) for i in range(1, n)}
    nodes = list(rng.permutation(n))
    try:
        ok_random &= _valid(topo_sort(nodes, par), nodes, par)
    except Exception:
        ok_random = False
check('5 random DAGs sorted correctly', bool(ok_random))

## Exercise 5 — Complete the autograd engine  `★★☆`

The `Value` class below is complete except for the backward closures of `*`, `exp`, `log`, `tanh` and `relu`.
Each closure receives `g`, the gradient of the loss with respect to the node's output, and must **add** the correct
contribution to each parent's `.grad` (use the local derivative rules in the concept guide's table). Note that the
forward values `e` and `t` are already available to the closures.

The checks compare your gradients with closed forms and with central differences; the hidden check differentiates
a composite expression through all five operations.

In [ ]:
class Value:
    def __init__(self, data, _parents=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._parents = _parents
        self._op = _op
        self._backward = lambda g: None

    def __repr__(self):
        return f"Value(data={self.data:.4g}, grad={self.grad:.4g})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward(g):
            self.grad += g
            other.grad += g
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward(g):
            self.grad += k * self.data ** (k - 1) * g
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward(g):
            # YOUR CODE HERE: add the contributions to self.grad and other.grad
            pass
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")
        def _backward(g):
            # YOUR CODE HERE
            pass
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")
        def _backward(g):
            # YOUR CODE HERE
            pass
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward(g):
            # YOUR CODE HERE
            pass
        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward(g):
            # YOUR CODE HERE
            pass
        out._backward = _backward
        return out

    def __neg__(self):             return self * -1.0
    def __radd__(self, other):     return self + other
    def __sub__(self, other):      return self + (-other)
    def __rsub__(self, other):     return other + (-self)
    def __rmul__(self, other):     return self * other
    def __truediv__(self, other):  return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1

    def topo_order(self):
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    visit(p)
                order.append(v)
        visit(self)
        return order

    def backward(self):
        self.grad = 1.0
        for v in reversed(self.topo_order()):
            v._backward(v.grad)

**Check your answer:**

In [ ]:
def _grads(fn, *vals):
    try:
        vs = [Value(v) for v in vals]
        out = fn(*vs)
        out.backward()
        return [v.grad for v in vs]
    except Exception:
        return None
def _numgrad(fn, *vals, h=1e-6):
    out = []
    for i in range(len(vals)):
        p = list(vals); m = list(vals); p[i] += h; m[i] -= h
        out.append((fn(*p) - fn(*m)) / (2 * h))
    return out
g_mul = _grads(lambda a, b: a * b, 3.0, -2.0)
check_close('mul: d(ab)/da = b and d(ab)/db = a', g_mul, [-2.0, 3.0])
check_close('mul: a*a gives 2a (two contributions)', _grads(lambda a: a * a, 1.5), [3.0])
check_close('exp: derivative is the output', _grads(lambda a: a.exp(), 0.7), [np.exp(0.7)])
check_close('log: derivative is 1/a', _grads(lambda a: a.log(), 4.0), [0.25])
check_close('tanh: derivative is 1 - tanh^2', _grads(lambda a: a.tanh(), 0.6), [1 - np.tanh(0.6) ** 2])
check_close('relu: 1 for positive, 0 for negative input', [_grads(lambda a: a.relu(), 2.0), _grads(lambda a: a.relu(), -2.0)], [[1.0], [0.0]])
composite = lambda a, b, c: ((a * b).exp() + c.log() * a.tanh()).relu() * (a * 2.0 - b) ** 2
num = _numgrad(lambda a, b, c: max(math.exp(a * b) + math.log(c) * math.tanh(a), 0.0) * (a * 2.0 - b) ** 2, 0.8, -0.3, 1.7)
g_comp = _grads(composite, 0.8, -0.3, 1.7)
check_close('composite expression matches central differences', g_comp, num, tol=1e-5)
check_hash('hidden: composite gradient', g_comp, '1e241541e030ff4e')

## Exercise 6 — Backpropagation through a two-layer network in NumPy  `★★☆`

Implement `mlp_grads(W1, b1, W2, b2, X, Y)` for a network with one $\tanh$ hidden layer and a linear output, trained
with the mean squared error over $N$ examples stored as **rows**:

$$Z_1 = X W_1^\top + b_1,\quad A_1 = \tanh(Z_1),\quad \hat Y = A_1 W_2^\top + b_2,\quad
\mathcal L = \frac{1}{2N}\sum_{n,k} (\hat Y_{nk} - Y_{nk})^2 .$$

Shapes: `X` is $(N, D)$, `W1` is $(H, D)$, `b1` is $(H,)$, `W2` is $(K, H)$, `b2` is $(K,)$, `Y` is $(N, K)$.
Return `loss, dW1, db1, dW2, db2` with the gradients shaped like the corresponding parameters. Use the recurrence
$\delta_2 = (\hat Y - Y)/N$, $\delta_1 = (\delta_2 W_2) \odot (1 - A_1^2)$, $\partial\mathcal L/\partial W_l = \delta_l^\top A_{l-1}$,
$\partial\mathcal L/\partial b_l = \sum_n \delta_l$.

In [ ]:
def mlp_grads(W1, b1, W2, b2, X, Y):
    # forward: Z1 = X W1^T + b1, A1 = tanh(Z1), Yhat = A1 W2^T + b2
    # loss = (1/(2N)) * sum((Yhat - Y)**2)
    # return loss, dW1, db1, dW2, db2 with the shapes of W1, b1, W2, b2
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
r6 = np.random.default_rng(6)
X6 = r6.normal(size=(5, 3)); Y6 = r6.normal(size=(5, 2))
W1 = r6.normal(size=(4, 3)) * 0.5; b1 = r6.normal(size=4) * 0.1
W2 = r6.normal(size=(2, 4)) * 0.5; b2 = r6.normal(size=2) * 0.1
try:
    loss, dW1, db1, dW2, db2 = mlp_grads(W1, b1, W2, b2, X6, Y6)
except Exception:
    loss = dW1 = db1 = dW2 = db2 = None
def _loss_only(W1_, b1_, W2_, b2_):
    A1 = np.tanh(X6 @ W1_.T + b1_)
    return 0.5 * np.sum((A1 @ W2_.T + b2_ - Y6) ** 2) / len(X6)
check_close('loss value', loss, _loss_only(W1, b1, W2, b2))
check('gradient shapes match the parameters',
      dW1 is not None and np.shape(dW1) == W1.shape and np.shape(db1) == b1.shape and np.shape(dW2) == W2.shape and np.shape(db2) == b2.shape)
def _num(param, setter, h=1e-6):
    g = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        p = param.copy(); m = param.copy(); p[idx] += h; m[idx] -= h
        g[idx] = (setter(p) - setter(m)) / (2 * h)
    return g
check_close('dW2 matches central differences', dW2, _num(W2, lambda P: _loss_only(W1, b1, P, b2)), tol=1e-6)
check_close('db2 matches central differences', db2, _num(b2, lambda P: _loss_only(W1, b1, W2, P)), tol=1e-6)
check_close('dW1 matches central differences', dW1, _num(W1, lambda P: _loss_only(P, b1, W2, b2)), tol=1e-6)
check_close('db1 matches central differences', db1, _num(b1, lambda P: _loss_only(W1, P, W2, b2)), tol=1e-6)
check_hash('hidden: sum of |dW1|', (float(np.sum(np.abs(dW1))) if dW1 is not None else None), 'c0ace32d2145c99a')

## Exercise 7 — The matmul backward rule  `★★☆`

For $Y = AB$ with $A \in \mathbb R^{m\times k}$, $B \in \mathbb R^{k \times n}$ and upstream gradient $G = \partial L/\partial Y \in \mathbb R^{m \times n}$,
implement `matmul_backward(A, B, G)` returning $(\bar A, \bar B) = (G B^\top,\ A^\top G)$.

Then write `numerical_matmul_grads(A, B, loss_fn, h=1e-6)` that estimates $\partial L/\partial A$ and $\partial L/\partial B$ entry by entry with
central differences for an arbitrary scalar `loss_fn(Y)` — this is the independent witness that lets you check the rule
for *any* downstream loss, not only $L = \sum Y$. Use `np.ndindex` to loop over entries.

In [ ]:
def matmul_backward(A, B, G):
    # Y = A @ B, G = dL/dY (same shape as Y). Return (dL/dA, dL/dB).
    # YOUR CODE HERE
    return None

def numerical_matmul_grads(A, B, loss_fn, h=1e-6):
    # central differences on every entry of A and of B for the scalar loss_fn(A @ B)
    # return (dA, dB) with the shapes of A and B
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
A = np.array([[1.0, 2.0], [3.0, 4.0]]); B = np.array([[1.0, 0.0], [1.0, 1.0]])
try:
    dA, dB = matmul_backward(A, B, np.ones((2, 2)))
except Exception:
    dA = dB = None
check_close('dA for L = sum(AB)', dA, np.array([[1.0, 2.0], [1.0, 2.0]]))
check_close('dB for L = sum(AB)', dB, np.array([[4.0, 4.0], [6.0, 6.0]]))
A2 = rng.normal(size=(3, 4)); B2 = rng.normal(size=(4, 2)); C2 = rng.normal(size=(3, 2))
loss_fn = lambda Y: float(np.sum((Y - C2) ** 2) + np.sum(np.sin(Y)))
try:
    Y2 = A2 @ B2
    G2 = 2 * (Y2 - C2) + np.cos(Y2)                 # dL/dY for this loss
    dA2, dB2 = matmul_backward(A2, B2, G2)
    nA, nB = numerical_matmul_grads(A2, B2, loss_fn)
except Exception:
    dA2 = dB2 = nA = nB = None
check('shapes: dA like A, dB like B', dA2 is not None and np.shape(dA2) == A2.shape and np.shape(dB2) == B2.shape)
check_close('rectangular case: dA matches central differences', dA2, nA, tol=1e-5)
check_close('rectangular case: dB matches central differences', dB2, nB, tol=1e-5)

## Exercise 8 — Fused, stable softmax cross-entropy  `★★☆`

Implement `log_softmax(Z)` for an $(N, K)$ array of logits using the log-sum-exp trick (subtract the row maximum before
exponentiating), and `softmax_ce(Z, labels)` returning the **mean** cross-entropy over rows,
$\mathcal L = -\frac1N\sum_n \log p_{n, y_n}$, together with its gradient $\partial\mathcal L/\partial Z = (P - Y)/N$ where $Y$ is the one-hot
matrix of `labels`. Your function must survive logits like 100 and -100 without `inf` or `nan`.

In [ ]:
def log_softmax(Z):
    # row-wise log-softmax of a (N, K) array, stable for large entries
    # YOUR CODE HERE
    return None

def softmax_ce(Z, labels):
    # mean cross-entropy over the N rows (labels: integer array of shape (N,)) and its gradient dL/dZ, shape (N, K)
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
Z8 = np.array([[2.0, 1.0, 0.1], [0.5, 0.5, 0.5], [100.0, 0.0, -100.0], [-3.0, 4.0, 1.0]])
lab8 = np.array([0, 2, 2, 1])
try:
    lp = log_softmax(Z8)
    loss8, grad8 = softmax_ce(Z8, lab8)
except Exception:
    lp = loss8 = grad8 = None
check('log-softmax rows exponentiate to probabilities summing to 1', lp is not None and np.allclose(np.exp(lp).sum(axis=1), 1.0))
check('no inf/nan for logits of +-100', lp is not None and np.all(np.isfinite(lp)) and loss8 is not None and np.isfinite(loss8))
check_close('uniform row has log-prob -log 3', (lp[1] if lp is not None else None), np.full(3, -np.log(3)))
def _num_grad(Z, h=1e-6):
    g = np.zeros_like(Z)
    for idx in np.ndindex(Z.shape):
        p = Z.copy(); m = Z.copy(); p[idx] += h; m[idx] -= h
        g[idx] = (softmax_ce(p, lab8)[0] - softmax_ce(m, lab8)[0]) / (2 * h)
    return g
try:
    ng = _num_grad(Z8)
except Exception:
    ng = None
check_close('gradient (P - Y)/N matches central differences', grad8, ng, tol=1e-6)
check('gradient rows sum to zero', grad8 is not None and np.allclose(grad8.sum(axis=1), 0.0))
check_hash('hidden: mean loss on the test logits', loss8, '4287ff789d9934c3')

## Exercise 9 — A reusable gradient checker  `★★☆`

Write `grad_check(loss_fn, grad_fn, theta, n_check=None, h=1e-5, rng=None)` where `loss_fn(theta)` returns a float and
`grad_fn(theta)` returns an array shaped like `theta`. Estimate the gradient entries by central differences (all entries,
or `n_check` randomly chosen ones using `rng.choice(theta.size, size=n_check, replace=False)`), and return the tuple
`(relative_error, worst_index)` with

$$\text{relative error} = \frac{\|g - \hat g\|_2}{\|g\|_2 + \|\hat g\|_2}, \qquad \text{worst index} = \arg\max_i |g_i - \hat g_i|$$

(the index into the flattened `theta`). The checker is used in the next two exercises, so make it robust to `theta` of
any shape.

In [ ]:
def grad_check(loss_fn, grad_fn, theta, n_check=None, h=1e-5, rng=None):
    # loss_fn(theta) -> float, grad_fn(theta) -> array like theta.
    # Check all entries (or n_check randomly chosen ones) with central differences and return
    # (relative_error, index_of_worst_entry), where relative_error = ||g - g_num|| / (||g|| + ||g_num||).
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
quad = lambda th: float(th @ th + np.sin(th[0]))
quad_grad = lambda th: 2 * th + np.array([np.cos(th[0]), 0.0, 0.0])
quad_grad_bad = lambda th: 2 * th + np.array([np.cos(th[0]), 0.0, 0.5])
th0 = np.array([0.3, -1.2, 2.0])
try:
    r_good, w_good = grad_check(quad, quad_grad, th0)
    r_bad, w_bad = grad_check(quad, quad_grad_bad, th0)
except Exception:
    r_good = r_bad = w_good = w_bad = None
check('correct gradient: relative error < 1e-7', r_good is not None and r_good < 1e-7)
check('wrong gradient: relative error > 1e-2', r_bad is not None and r_bad > 1e-2)
check('worst index points at the wrong entry (2)', w_bad == 2)
M = rng.normal(size=(3, 3)); M = M + M.T
quad2 = lambda th: float(0.5 * th.ravel() @ M @ th.ravel())
quad2_grad = lambda th: (M @ th.ravel()).reshape(th.shape)
try:
    r_sub, _ = grad_check(quad2, quad2_grad, np.ones((3, 1)), n_check=2, rng=np.random.default_rng(3))
except Exception:
    r_sub = None
check('2-D theta and a random subset of entries', r_sub is not None and r_sub < 1e-7)

## Exercise 10 — Forward or reverse mode?  `★★★`

For each situation, say whether you would use forward mode or reverse mode, roughly how many sweeps it costs, and why:

1. The gradient of a scalar training loss for a network with 10 million parameters.
2. The full Jacobian of a function $f: \mathbb R^{3} \to \mathbb R^{5000}$ (3 design parameters, 5000 simulated sensor readings).
3. A Hessian-vector product $Hv$ for the loss in (1).
4. A colleague proposes estimating the gradient in (1) with central differences on a GPU "because it parallelises well". What is wrong with that?

<details><summary><b>Show answer</b></summary>

1. Reverse mode: one output, so a single backward sweep (a small constant times one forward pass) gives all $10^7$ partial
   derivatives. Forward mode would need $10^7$ sweeps.
2. Forward mode: three inputs, so three sweeps (one per input, each seeding a different tangent) give the entire
   $5000\times3$ Jacobian. Reverse mode would need 5000 sweeps, one per output. Dual numbers or `jax.jvp` are the tools.
3. Forward-over-reverse: compute the gradient $\nabla L$ with one reverse sweep, then take a forward-mode (directional)
   derivative of that gradient in direction $v$. Cost is about two to four forward-pass equivalents, independent of the
   parameter count, and the $10^7\times10^7$ Hessian is never formed (Pearlmutter's trick).
4. Parallelism does not change the operation count: $2\times10^7$ forward passes per gradient versus roughly three for
   backpropagation, a factor of about $10^7$ more work per step. The estimate is also only approximate — truncation
   and round-off errors, severe in float32 — while reverse mode is exact to rounding. Finite differences are for
   *checking* a few entries, not for training.

</details>

## Exercise 11 — Diagnose a planted bug  `★★★`

The function below computes the loss and gradients of a one-layer $\tanh$ regression model,
$A = \tanh(X W^\top + b)$, $\mathcal L = \tfrac12\,\mathrm{mean}\big((A - Y)^2\big)$. One line of its backward pass is wrong.

```python
def buggy_forward_backward(W, b, X, Y):
    Z = X @ W.T + b
    A = np.tanh(Z)
    loss = 0.5 * np.mean((A - Y) ** 2)
    G = (A - Y) / A.size
    dZ = G * (1.0 - Z ** 2)
    dW = dZ.T @ X
    db = dZ.sum(axis=0)
    return loss, dW, db
```

1. Use your `grad_check` from Exercise 9 on `buggy_forward_backward` (flatten `W` and `b` into one parameter vector) at
   two scales of `W` — entries of size about 0.1 and about 2 — and note how the relative error depends on the scale.
2. Set `bug_location` to one of `"forward pass"`, `"loss"`, `"upstream gradient G"`, `"tanh derivative"`, `"dW"`, `"db"`.
3. Write `fixed_forward_backward` with the repair. The checks run your gradient checker on it.

In [ ]:
def buggy_forward_backward(W, b, X, Y):
    # a tanh regression layer with a planted bug in the backward pass
    Z = X @ W.T + b
    A = np.tanh(Z)
    loss = 0.5 * np.mean((A - Y) ** 2)
    G = (A - Y) / A.size
    dZ = G * (1.0 - Z ** 2)            # <-- derivative of tanh written in terms of the input
    dW = dZ.T @ X
    db = dZ.sum(axis=0)
    return loss, dW, db

def fixed_forward_backward(W, b, X, Y):
    # copy buggy_forward_backward here and repair the one wrong line
    # YOUR CODE HERE
    return None

# one of: "forward pass", "loss", "upstream gradient G", "tanh derivative", "dW", "db"
bug_location = None

**Check your answer:**

In [ ]:
r11 = np.random.default_rng(11)
X11 = r11.normal(size=(6, 3)); Y11 = r11.uniform(-0.9, 0.9, size=(6, 2))
W11 = r11.normal(size=(2, 3)); b11 = r11.normal(size=2) * 0.1
theta11 = np.concatenate([W11.ravel(), b11])
def _unpack(th):
    return th[:6].reshape(2, 3), th[6:]
try:
    loss_fixed = lambda th: fixed_forward_backward(*_unpack(th), X11, Y11)[0]
    grad_fixed = lambda th: np.concatenate([fixed_forward_backward(*_unpack(th), X11, Y11)[1].ravel(),
                                            fixed_forward_backward(*_unpack(th), X11, Y11)[2]])
    r_fixed, _ = grad_check(loss_fixed, grad_fixed, theta11)
    same_loss = abs(loss_fixed(theta11) - buggy_forward_backward(W11, b11, X11, Y11)[0]) < 1e-12
    grad_buggy = lambda th: np.concatenate([buggy_forward_backward(*_unpack(th), X11, Y11)[1].ravel(),
                                            buggy_forward_backward(*_unpack(th), X11, Y11)[2]])
    r_buggy, _ = grad_check(loss_fixed, grad_buggy, theta11)
except Exception:
    r_fixed = r_buggy = None; same_loss = False
check('the buggy version fails the gradient check (relative error > 1e-2)', r_buggy is not None and r_buggy > 1e-2)
check('the fixed version passes the gradient check (relative error < 1e-7)', r_fixed is not None and r_fixed < 1e-7)
check('the fix leaves the forward pass / loss unchanged', bool(same_loss))
check_hash('bug location named correctly', bug_location, '476bdba03106da04')

## Exercise 12 — Mini-project: train a network with your own engine  `★★★`

Using the `Value` class you completed in Exercise 5 (it is still defined in this notebook), build and train a small
neural network on the two-moons dataset:

```python
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=200, noise=0.15, random_state=0)
```

1. Write `Neuron`, `Layer` and `MLP` classes made of `Value`s (ReLU or tanh hidden units, a linear output logit), with a
   `parameters()` method. A 2–16–1 architecture (65 parameters) is enough.
2. Implement a numerically stable binary cross-entropy on the logit, either as a fused `Value` operation with gradient
   $\sigma(s) - y$ or by composing `Value` operations in the log-sum-exp form.
3. Before training, gradient-check a handful of parameters with central differences (relative error below $10^{-6}$).
4. Train with minibatch SGD or Adam for a few hundred steps, remembering to zero the gradients before each backward
   pass. Keep the run under about 30 seconds: a 2–16–1 network, minibatches of 32 and 300 steps are plenty.
5. Plot the training loss curve and the decision boundary (evaluate the model on a grid and use `plt.contourf`), and
   report the final training accuracy. Aim for **above 95 percent**.

**Rubric — you have mastered this when you can:**

- Your network is built only from `Value` objects and your own code (no NumPy autodiff tricks, no frameworks).
- A gradient check on at least 5 parameters passes with relative error below $10^{-6}$ before training starts.
- Gradients are zeroed before every backward pass, and the loss is averaged over the minibatch.
- Training accuracy exceeds 95 percent and the loss curve decreases (noisy minibatch loss is fine).
- The decision boundary plot shows a curved boundary separating the two moons, with axis labels and a title.
- You can explain why the fused `bce_with_logits` is preferable to `-(y*log(p) + (1-y)*log(1-p))` built from `log` and `exp` nodes.

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=200, noise=0.15, random_state=0)

# Your mini-project workspace: Neuron / Layer / MLP, a stable BCE, a gradient check, a training loop, two plots.

## Score

In [ ]:
summary()